<a href="https://colab.research.google.com/github/zyf-hitsz/PytorchLearning/blob/main/%E7%A5%9E%E7%BB%8F%E7%BD%91%E7%BB%9C/%E5%BD%92%E4%B8%80%E5%8C%96%E5%B1%82/NormalizationLayers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#归一化层NormalizationLayers


## 归一化层 (Normalization Layers) 的目的和作用

归一化层是深度学习中常用的技术，其主要目的是解决训练过程中“内部协变量偏移”（Internal Covariate Shift）的问题，从而加速模型的收敛，提高模型的稳定性和性能。

### 内部协变量偏移 (Internal Covariate Shift)

在深度神经网络中，每个层的输入都受到前面所有层的参数变化的影响。这意味着在训练过程中，即使我们保持输入数据的分布不变，每一层所接收到的输入数据的分布也会不断变化。这种现象被称为“内部协变量偏移”。

这种变化会导致以下问题：

1.  **训练困难**：每层需要不断适应新的输入分布，导致训练过程变得不稳定，需要更小的学习率。
2.  **梯度消失/爆炸**：当输入分布变化剧烈时，可能导致激活函数饱和，梯度变得非常小（消失）或非常大（爆炸）。
3.  **收敛速度慢**：模型需要更多的时间和迭代才能收敛。

### 归一化层的目的

归一化层通过将每一层输入的分布规范化到稳定的范围（通常是零均值和单位方差），来缓解内部协变量偏移问题。这样，每一层在训练时面对的输入分布就更加一致，从而达到以下目的：

1.  **加速训练收敛**：通过减少内部协变量偏移，模型可以使用更大的学习率，从而加快训练速度。
2.  **提高模型稳定性**：规范化的输入使得梯度更加稳定，避免了梯度消失或爆炸的问题。
3.  **改善模型泛化能力**：有助于模型更好地学习数据的特征，减少过拟合的风险。
4.  **减少对初始化权重的依赖**：模型对初始权重的选择不那么敏感。

### 归一化层的作用

归一化层在深度学习中扮演着至关重要的角色，具体作用体现在：

*   **标准化输入**：将输入数据调整到均值为0、方差为1的标准正态分布，确保数据在激活函数输入时处于非饱和区域。
*   **平滑损失函数**：使得损失函数的曲面更加平滑，有利于优化器找到全局最优解。
*   **正则化效果**：在一定程度上具有正则化效果，类似于 Dropout，减少模型对训练数据的依赖，降低过拟合风险。
*   **允许更大的学习率**：由于梯度更稳定，可以安全地使用更大的学习率，加速训练过程。

##BatchNormNd
核心思想：对同一个 channel，跨 batch 中的多个样本统计均值和方差。

In [ ]:
#以二维为例
class torch.nn.BatchNorm2d(num_features, eps=1e-05, momentum=0.1,
              affine=True, track_running_stats=True, device=None, dtype=None, *, bias=True)
#num_features:输入通道数
#eps:默认为1e-05，为了数值稳定性，在方差分母上加一个极小的值，防止除以零。
#mmentum:用于计算动态均值和动态方差的动量值，默认为0.1.
#affine:布尔值，当设为 True 时，该层会学习两个可学习的仿射变换参数：γ(weight)和β(bias)。
        #它们的作用是将归一化后的数据进行缩放和平移，以恢复网络的表达能力。
#track_running_stats:布尔值。
        #当设为 True 时，模型会跟踪并保存训练期间的均值和方差，以便在测试（eval）模式下使用。
        #如果设为 False，则在训练和测试时都只使用当前 batch 的统计数据。
#bias：如果 affine 为 True，该参数控制是否包含可学习的偏置项β。

例如：

$$X\in R^{N\times C\times H\times W}$$
对于第 $c $个 channel：

$$\mu_c=
\frac{1}{NHW}
\sum_{n,h,w}x_{n,c,h,w}
$$方差：
$$
\sigma_c^2=
\frac{1}{NHW}
\sum_{n,h,w}
(x_{n,c,h,w}-\mu_c)^2
$$然后：
$$
\hat{x}_{n,c,h,w}
=
\frac{x_{n,c,h,w}-\mu_c}
{\sqrt{\sigma_c^2+\epsilon}}
$$最终：
$$
y_{n,c,h,w}
=
\gamma_c\hat{x}_{n,c,h,w}+\beta_c
$$注意：
$$
\gamma,\beta\text{通常每个 channel 各一组}
$$

### BatchNorm 的训练与推理差异

BatchNorm 在训练和推理阶段表现不同，这是为了解决推理时可能只有一个样本（Batch Size = 1）而无法计算有效均值和方差的问题。

#### 1. 训练阶段 (Training)
*   **统计量来源**：使用当前 Batch 数据的均值 $\mu_B$ 和方差 $\sigma_B^2$ 进行归一化。
*   **状态更新**：在训练过程中，BatchNorm 会维护两个“影子”变量：**Running Mean (运行均值)** 和 **Running Variance (运行方差)**。它们通过动量（momentum）机制不断更新：
    $$\text{running_mean} = (1 - \text{momentum}) \times \text{running_mean} + \text{momentum} \times \mu_B$$
    $$\text{running_var} = (1 - \text{momentum}) \times \text{running_var} + \text{momentum} \times \sigma_B^2$$

#### 2. 推理/测试阶段 (Inference/Evaluation)
*   **切换模式**：必须调用 `model.eval()`。此时 BatchNorm 层会停止更新 Running Stats，并进入固定模式。
*   **统计量来源**：**不再计算**当前 Batch 的均值和方差，而是直接使用训练阶段累积下来的 `running_mean` 和 `running_var`。
*   **目的**：确保模型对于相同的输入，无论 Batch Size 是多少（甚至是单个样本），其输出结果都是确定且一致的。

#### 3. 为什么必须区分？
*   **稳定性**：推理时的数据分布可能与训练 Batch 差异较大，使用训练好的全局统计量更具代表性。
*   **独立性**：预测单个样本时，模型不应该依赖于同批次的其他样本。
---
BatchNorm 的优点与缺点：

优点：
*   训练稳定
*   可以使用更大的学习率
*   对初始化更宽容
*   有一定正则化效果

缺点：
*   对batchsize比较敏感，batchsize较小时可能非常不准








##Layer Normalization
一个样本内部，跨特征进行统计。

In [ ]:
class torch.nn.LayerNorm(normalized_shape, eps=1e-05,
             elementwise_affine=True,bias=True, device=None, dtype=None)
#normalized_shape:指定要进行归一化的维度形状。
#如果输入是 (N, *)，则 normalized_shape 必须是输入形状的后半部分。
#例如，如果输入形状是 (batch, time, channels)，你可以设置 normalized_shape为[channels]或[time, channels]。

### LayerNorm 的特点
- **不依赖 Batch Size**：每个样本的归一化都是独立计算的，因此在训练和推理（测试）阶段的行为是**一致**的。这也是它在处理 NLP 任务或小 Batch 情况时优于 BatchNorm 的原因。
- **LayerNorm训练阶段和推理阶段基本一致**

###Transformer中最典型的LayerNorm
例如 Transformer：
$$
X\in R^{B\times L\times D}
$$其中：

$B$：batch size

$L$：sequence length

$D$：embedding dimension

例如：
[32, 128, 768]

表示：
32 个句子;每个句子 128 token;每个 token 是 768 维向量

LayerNorm 通常针对：
$$
D=768
$$进行归一化。
也就是对于某个 token：
$$
x=
[x_1,x_2,\cdots,x_{768}]
$$计算：
$$
\mu=
\frac1{768}
\sum_{i=1}^{768}x_i
$$
$$
\sigma^2=
\frac1{768}
\sum_{i=1}^{768}(x_i-\mu)^2
$$然后：
$$
\hat{x}_i=
\frac{x_i-\mu}
{\sqrt{\sigma^2+\epsilon}}
$$所以：
$$
\boxed{
每个token自己归一化自己的768维feature
}
$$和 batch 中有多少样本完全无关。

因为Transformer中的batch size和sequence length经常变化，如果使用BatchNorm，统计量会受到batch组成影响，因此选择分batch处理的LayerNorm。

In [4]:
import torch
import torch.nn as nn

# 定义 LayerNorm 层，针对特征维度 3
norm = nn.LayerNorm(3)

# 创建输入数据 (Batch Size=2, Sequence Length=4, Embedding Dim=3)
x = torch.randn(2, 4, 3)

# 进行归一化
y = norm(x)

print(f"输入: {x}")
print(f"输出: {y}")

输入: tensor([[[-1.5997, -0.2181,  0.3531],
         [-0.1384,  0.3148, -0.5392],
         [-1.3237, -2.5157, -0.5452],
         [-1.7684,  0.6104, -1.1029]],

        [[ 0.6353, -0.2791,  0.2928],
         [ 0.7177,  0.2175, -2.0863],
         [ 0.8597, -1.1730,  0.8093],
         [-0.9167,  0.6341, -0.4964]]])
输出: tensor([[[-1.3558,  0.3295,  1.0263],
         [-0.0500,  1.2489, -1.1989],
         [ 0.1701, -1.3009,  1.1308],
         [-1.0127,  1.3612, -0.3485]],

        [[ 1.1107, -1.3134,  0.2027],
         [ 0.9019,  0.4923, -1.3943],
         [ 0.7336, -1.4139,  0.6803],
         [-1.0033,  1.3648, -0.3615]]], grad_fn=<NativeLayerNormBackward0>)


##Group Normalization
Group Normalization可以理解成BatchNorm和LayerNorm之间的一种折中。它最初主要是为了解决CNN中小 batch 时 BatchNorm 不稳定的问题。


In [ ]:
class torch.nn.GroupNorm(num_groups, num_channels, eps=1e-05,
                  affine=True, device=None, dtype=None, *, bias=True)
#num_groups (G):这是一个核心参数，所有通道会被平分为 G 组。
    #注意：num_channels 必须能被 num_groups 整除。
#num_channels=1时，等价为Layer Normalization ；num_channels=num_channels时，等价为Instance Normalization

$$
X\in R^{N\times C\times H\times W}
$$例如：
N = 8
C = 64
H = 32
W = 32
假设：
nn.GroupNorm(8, 64)
表示：
$$
64\text{ channels}
$$分成：
$8\text{ groups}$

每组：$64/8=8$个 channel。

然后：

Group 1:
channel 0~7

Group 2:
channel 8~15

...

Group 8:
channel 56~63

每个 group 内部，对：
$$
C_{\text{group}}\times H\times W
$$计算：
$$
\mu,\sigma^2
$$

这使得它既不像 BN 那样依赖 Batch Size，又比 LN 保留了更多的通道差异性。

注意：
$$GroupNorm不跨batch$$

##RMSNorm——均方根归一化
$$\text{Root Mean Square Normalization}
，$$Transformer / 大语言模型中非常重要的一种 Norm，即均方根归一化。可以理解为LinearNorm的简化版。
RMSNorm 认为LayerNorm 中“减均值”这一步并不一定必要。

因此直接计算：
$$
RMS(x)=
\sqrt{
\frac1d
\sum_{i=1}^d x_i^2
+\epsilon
}
$$然后：
$$
\hat{x}_i=
\frac{x_i}{RMS(x)}
$$最终：
$$
y_i=
\gamma_i\hat{x}_i
$$很多实现：
$$
\boxed{\text{没有 }\beta}
$$所以它不做：
$$
x-\mu
$$只做：
$$
\boxed{\text{控制向量的整体尺度}}
$$
LinearNorm修改了向量中心，而RMSNorm不修改向量中心，只改变尺度。RMSNorm结构更简单，计算更少，训练效果往往更好。
